# Notebook 03 - Unified Runner + Root Agent + Verifier Demo

This notebook unifies the end-to-end verification story in one executable flow:

1. Load a deterministic scenario from `workshop_verify_agents.examples.conversations`.
2. Run multi-turn execution with `ADKConversationRunner` when available.
3. Fall back to synthetic orchestrator runtime when ADK credentials are not available.
4. Run online turn-level behaviour verification for each turn.
5. Run offline conversation-level unified verification with journey-aware outcome evidence.

The default scenario is `mortgage_hidden_eligibility_failure`, chosen because it clearly shows why conversation-level outcome verification matters.

In [ ]:
# Cell 1 - Setup imports and runtime guards
from __future__ import annotations

from pathlib import Path
from uuid import uuid4
import sys

import pandas as pd
from IPython.display import Markdown, display

# Ensure src/ is importable when notebook is launched from nested paths.
CWD = Path.cwd().resolve()
SRC_CANDIDATES = [
    CWD / 'src',
    CWD.parent / 'src',
    CWD.parent.parent / 'src',
    CWD.parent.parent.parent / 'src',
]
for candidate in SRC_CANDIDATES:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from workshop_verify_agents.examples.conversations import get_scenario, list_scenarios
from workshop_verify_agents.synthetic_runtime.orchestrator import run as run_synthetic_turn
from workshop_verify_agents.verification.online_verifier import verify_turn_behaviour
from workshop_verify_agents.verification.offline_verifier import verify_completed_conversation

ADK_AVAILABLE = True
ADK_IMPORT_ERROR = None
try:
    from workshop_verify_agents.adk_runtime.runner import ADKConversationRunner
    from workshop_verify_agents.adk_runtime.root_agent import root_agent
except Exception as exc:
    ADK_AVAILABLE = False
    ADK_IMPORT_ERROR = exc

display(Markdown('### Runtime availability'))
if ADK_AVAILABLE:
    print('ADK runner: available')
    print('Root agent name:', root_agent.name)
    print('Root agent sub-agent count:', len(root_agent.sub_agents))
else:
    print('ADK runner: unavailable (synthetic fallback will be used)')
    print(f'Reason: {type(ADK_IMPORT_ERROR).__name__}: {ADK_IMPORT_ERROR}')
print('Synthetic orchestrator: available')

### Runtime availability

ADK runner: available
Root agent name: finance_verification_agent
Root agent sub-agent count: 6
Synthetic orchestrator: available


In [2]:
# Cell 2 - Choose scenario and inspect conversation plan
SCENARIO_NAME = 'mortgage_hidden_eligibility_failure'
scenario = get_scenario(SCENARIO_NAME)

display(Markdown(f'## Scenario: `{SCENARIO_NAME}`'))
print('Description:', scenario['description'])

scenario_table = pd.DataFrame([
    {
        'turn_index': idx,
        'role': turn['role'],
        'text': turn['text'],
        'expected_stage': scenario['expected_stage_labels'][idx],
        'mocked_quality_score': scenario['mocked_quality_scores'][idx],
    }
    for idx, turn in enumerate(scenario['turns'])
])
display(scenario_table)

available = list_scenarios()
print('Available scenarios:', ', '.join(available))

## Scenario: `mortgage_hidden_eligibility_failure`

Description: Conversation appears fluent overall but performs poorly at eligibility checks.


,turn_index,role,text,expected_stage,mocked_quality_score
0,0,user,"Hello, can you help me plan for a mortgage thi...",greeting,0.92
1,1,agent,Certainly. We can walk through borrowing and r...,discovery,0.86
2,2,user,I have variable income and some debt. Is that ...,clarification,0.83
3,3,agent,You should still be fine. Let us jump to optio...,eligibility_or_suitability,0.45
4,4,user,Which route do you suggest?,recommendation_or_guidance,0.88
5,5,agent,"A fixed-rate option could be convenient, then ...",closing,0.90


Available scenarios: mortgage_happy_path, mortgage_hidden_eligibility_failure, investment_good_journey, investment_suitability_failure, pension_recovery_journey


In [4]:
# Cell 3 - Execute multi-turn runtime using scenario user turns
conversation_id = f'unified-demo-{uuid4()}'
user_turns = [t['text'] for t in scenario['turns'] if t['role'] == 'user']

runtime_mode = 'synthetic'
turn_outputs = []

if ADK_AVAILABLE:
    runtime_mode = 'adk_runner'
    runner = ADKConversationRunner()

if runtime_mode == 'adk_runner':
    for idx, query in enumerate(user_turns):
        response_text, _raw_events, trace = await runner.run_turn(query, conversation_id)
        turn_outputs.append({
            'turn_index': idx,
            'query': query,
            'response': response_text,
            'trace': trace,
        })
else:
    for idx, query in enumerate(user_turns):
        synthetic_result = run_synthetic_turn(
            query,
            conversation_id=conversation_id,
            turn_id=str(uuid4()),
        )
        response_text, trace = synthetic_result.final_response, synthetic_result.execution_trace
        turn_outputs.append({
            'turn_index': idx,
            'query': query,
            'response': response_text,
            'trace': trace,
        })

display(Markdown(f'### Runtime mode used: `{runtime_mode}`'))
runtime_table = pd.DataFrame([
    {
        'turn_index': item['turn_index'],
        'query': item['query'],
        'response_preview': item['response'][:180],
        'trace_event_count': len(item['trace'].events),
    }
    for item in turn_outputs
])
display(runtime_table)

### Runtime mode used: `adk_runner`

,turn_index,query,response_preview,trace_event_count
0,0,"Hello, can you help me plan for a mortgage thi...",It's great that you're getting organised this ...,7
1,1,I have variable income and some debt. Is that ...,I can certainly help you with questions about ...,7
2,2,Which route do you suggest?,"I can help with questions about mortgages, inv...",7


In [5]:
# Cell 4 - Online behaviour verification per turn
behaviour_results = []

for item in turn_outputs:
    result = verify_turn_behaviour(
        conversation_id=conversation_id,
        turn_index=item['turn_index'],
        execution_trace=item['trace'],
    )
    behaviour_results.append(result)

online_table = pd.DataFrame([
    {
        'turn_index': r.turn_index,
        'signal_count': len(r.business_signals),
        'behaviour_evidence_count': len(r.behaviour_evidence),
        'behaviour_status': r.behaviour_assessment.overall_status.value,
        'warnings': len(r.behaviour_assessment.warnings),
        'failures': len(r.behaviour_assessment.failures),
    }
    for r in behaviour_results
])

display(Markdown('## Online lane summary (turn-level behaviour verification)'))
display(online_table)

## Online lane summary (turn-level behaviour verification)

,turn_index,signal_count,behaviour_evidence_count,behaviour_status,warnings,failures
0,0,5,3,pass,0,0
1,1,5,3,pass,0,0
2,2,5,3,pass,0,0


In [8]:
# Cell 5 - Offline unified verification (conversation-level behaviour + outcome)
unified = verify_completed_conversation(
    conversation_id=conversation_id,
    behaviour_results=behaviour_results,
    conversation_turns=scenario['turns'],
    turn_scores=scenario['mocked_quality_scores'],
    journey_stage_labels=scenario['expected_stage_labels'],
)

final_assessment = unified.final_assessment
outcome = unified.outcome_result

overview = pd.DataFrame([
    {
        'conversation_id': unified.conversation_id,
        'completeness': unified.completeness,
        'final_status': final_assessment.overall_status.value,
        'warning_count': len(final_assessment.warnings),
        'failure_count': len(final_assessment.failures),
        'offline_warnings': ' | '.join(unified.warnings) if unified.warnings else '',
    }
])

display(Markdown('## Offline unified summary (conversation level)'))
display(overview)

if outcome is not None:
    simple = outcome.journey_aware_result.get('simple_average')
    if simple is None:
        simple = outcome.journey_aware_result.get('simple_average_score')

    weighted = outcome.journey_aware_result.get('journey_weighted_score')
    comparison = pd.DataFrame([
        {
            'simple_average_score': simple,
            'journey_weighted_score': weighted,
            'score_delta': outcome.journey_aware_result.get('score_delta'),
            'simple_rating': outcome.journey_aware_result.get('simple_rating'),
            'journey_weighted_rating': outcome.journey_aware_result.get('journey_weighted_rating'),
        }
    ])
    display(Markdown('### Outcome lane comparison'))
    display(comparison)

## Offline unified summary (conversation level)

,conversation_id,completeness,final_status,warning_count,failure_count,offline_warnings
0,unified-demo-a2e40b81-f73b-4643-8269-e3e0592ec964,offline_full,pass_with_warnings,1,0,


### Outcome lane comparison

,simple_average_score,journey_weighted_score,score_delta,simple_rating,journey_weighted_rating
0,0.8067,0.7224,-0.0843,PASS,WARNING


In [7]:
# Cell 6 - Optional drill-down for one selected turn
TURN_TO_INSPECT = 0
selected = behaviour_results[TURN_TO_INSPECT]

signals_df = pd.DataFrame([
    {
        'signal_name': s.signal_name,
        'signal_type': s.signal_type.value,
        'severity': s.severity.value,
        'confidence': s.confidence,
        'explanation': s.explanation,
    }
    for s in selected.business_signals
])

evidence_df = pd.DataFrame([
    {
        'evidence_type': e.evidence_type.value,
        'status': e.status.value,
        'related_signal_count': len(e.related_signals),
        'explanation': e.explanation,
    }
    for e in selected.behaviour_evidence
])

display(Markdown(f'## Turn drill-down: {TURN_TO_INSPECT}'))
display(Markdown('### Signals'))
display(signals_df)
display(Markdown('### Behaviour evidence'))
display(evidence_df)
print('Assessment summary:', selected.behaviour_assessment.summary)

## Turn drill-down: 0

### Signals

,signal_name,signal_type,severity,confidence,explanation
0,safety_compliance,safety,low,1.0,Safety layer cleared the request without inter...
1,intent_routing,routing,low,1.0,Intent classified as 'mortgage' with confidenc...
2,journey_routing,journey,low,1.0,Conversation routed to the 'mortgage' domain a...
3,execution_path,execution,low,1.0,Sub-agent 'mortgage' invoked for topic 'unknown'.
4,outcome_availability,outcome,low,1.0,A response of 4097 characters was delivered to...


### Behaviour evidence

,evidence_type,status,related_signal_count,explanation
0,behaviour,pass,2,Agent routing and journey behaviour passed. Si...
1,governance,pass,1,Safety and governance controls passed. Signals...
2,operational,pass,1,Execution and operational reliability passed. ...


Assessment summary: The conversation passed all verification checks.  Evidence dimensions assessed: [behaviour=pass, governance=pass, operational=pass].


## Notes

- This notebook demonstrates both verifier lanes in one place.
- Online lane stays behaviour-only at turn level.
- Offline lane adds journey-aware outcome evidence at conversation level.
- If ADK credentials are not configured, synthetic runtime still demonstrates the same verification interfaces.